In [2]:
import os
import pickle
import gc
from tqdm import tqdm
import time
import polars as pl
import numpy as np
import shutil
import glob
from scipy import sparse
from tienlen_libs import Card, Utils, ActionGenerator, NaturalWin
from tienlen_libs.constant import ComboType
import math
from collections import defaultdict

In [3]:
def check_data(list_file_src_dir):
    list_who_play_first = []
    set_player1_card = set()
    for file_dir in tqdm(list_file_src_dir):
        try:
            with open(file_dir, 'rb') as f:
                data = pickle.load(f)
            list_who_play_first.append(data["who_play_first"])
            player1_card = tuple(data["player_card"][1])
            set_player1_card.add(player1_card)
            for state in data["Q_table"]:
                player_action = state[1]
                assert len(player_action) == 2
                player2_action = player_action[0]
                player3_action = player_action[1]
                turn = state[2]
                if turn == 1:
                    for player_id_action in player_action:
                        assert player_id_action == tuple() or player_id_action == -1
                elif turn == 2:
                    assert isinstance(player2_action, tuple) and len(player2_action) > 0
                    assert player3_action == tuple() or player3_action == -1
                elif turn == 3:
                    assert isinstance(player3_action, tuple) and len(player3_action) > 0

                player_card = state[0]
                for i in range(1, 4):
                    assert set(player_card[i-1]).issubset(set(data["player_card"][i]))
        except BaseException as e:
            print(f"ERROR with file {file_dir}")
            print(e)
    return list_who_play_first, set_player1_card

In [ ]:
def generate_one_hot_encoding(cards: tuple[int], dimension: int = 52):
    return tuple(True if i in cards else False for i in range(dimension))

def generate_action_encoding(action: tuple[int]):
    action_int_combo_type = Utils.get_int_combo_type(action)
    action_int_combo_type_encoding = generate_one_hot_encoding((action_int_combo_type,),
                                                               dimension=19)
    assert action_int_combo_type_encoding.count(True) == 1        
    max_card = max(action) if len(action) > 0 else -1
    return action_int_combo_type_encoding + (max_card,)

def generate_list_state_action_return_encode(file_data_dir):
    with open(file_data_dir, "rb") as f:
        game_data = pickle.load(f)
    print(f"game_data = {game_data}")
    Q_table = game_data["Q_table"]

    # for state in Q_table:
    #     if len(Q_table[state]) <= 1: continue
    #     player2_card, player1_action, actions_card_player1, actions_card_player2, deck_remain, len_player1_card = state

    #     player2_card_encoding = generate_one_hot_encoding(player2_card)
    #     player1_action_encoding = generate_action_encoding(player1_action)
    #     player1_action_one_hot_encoding = generate_one_hot_encoding(player1_action)
    #     actions_card_player1_encoding = generate_one_hot_encoding(actions_card_player1)
    #     actions_card_player2_encoding = generate_one_hot_encoding(actions_card_player2)
    #     deck_remain_encoding = generate_one_hot_encoding(deck_remain)

    #     for action in Q_table[state]:
    #         expected_return = Q_table[state][action]
    #         assert 0 <= expected_return <= 1

    #         action_encoding = generate_action_encoding(action)
    #         action_one_hot_encoding = generate_one_hot_encoding(action)

    #         assert set(action).issubset(set(player2_card))
    #         player2_card_remain = tuple([card for card in player2_card if card not in action])
    #         player2_card_remain_encoding = generate_one_hot_encoding(player2_card_remain)

    #         state_action_return_encode = player2_card_encoding  + \
    #                                     player1_action_encoding + player1_action_one_hot_encoding + \
    #                                     actions_card_player1_encoding + actions_card_player2_encoding + \
    #                                     deck_remain_encoding + \
    #                                     (len_player1_card,) + action_encoding + action_one_hot_encoding + \
    #                                     player2_card_remain_encoding + (player2_play_first, expected_return)
    #         yield state_action_return_encode


file_data_dir = r"D:\Project\TLMN\data_see_opponent_card_3_players\TLMN_2025-06-09-21-34-41_CP24027HAYDEN_22860_185.pkl"
generate_list_state_action_return_encode(file_data_dir)